#1. Install deps

In [3]:
import sys
import os
import json
import pandas as pd

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/NLP'):
        !git clone -b lab-14 https://github.com/AndrianaNahirna/NLP.git

    %cd /content/NLP

    !pip install -r requirements.txt -q
    !git fetch origin
    !git reset --hard origin/lab-14

    sys.path.append('/content/NLP')

print("Середовище налаштовано.")

Cloning into 'NLP'...
remote: Enumerating objects: 496, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 496 (delta 4), reused 13 (delta 4), pack-reused 418 (from 1)
Receiving objects: 100% (496/496), 2.32 MiB | 4.18 MiB/s, done.
Resolving deltas: 100% (238/238), done.
/content/NLP
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.1/990.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 33.0 MB/s eta 0:00:00
HEAD is now at b1d9dd3 add src
Середовище налаштовано.


#2. Test cases

In [1]:
test_cases = [
    # 1. Простий кейс, де flow проходить усі етапи без помилок
    {
        "case_id": "tc_001",
        "input": "Моя оплата за підписку не пройшла, гроші списало.",
        "expected_route": "support_classification",
        "expected_behavior": "extract class as Billing with high confidence, validation ok, export ok"
    },

    # 2. Кейс із missing required field
    {
        "case_id": "tc_002",
        "input": "оплата",
        "expected_route": "support_classification",
        "expected_behavior": "validate catches missing 'confidence' field, trigger warning"
    },

    # 3. Кейс із unknown route
    {
        "case_id": "tc_003",
        "input": "Яка завтра погода у Львові?",
        "expected_route": "unknown",
        "expected_behavior": "router assigns unknown, execute skips, safely fails"
    },

    # 4. Кейс, де validation ловить проблему (Low confidence)
    {
        "case_id": "tc_004",
        "input": "Щось пішло не так з моїм акаунтом.",
        "expected_route": "support_classification",
        "expected_behavior": "classifier gives low confidence (<0.7), validation catches it and fails"
    },

    # 5. Кейс, де потрібен fallback
    {
        "case_id": "tc_005",
        "input": "Питання щодо профілю",
        "expected_route": "support_classification",
        "expected_behavior": "validation fails due to low confidence, strictly triggers fallback logic"
    },

    # 6. Кейс, де fallback допомагає (Recoverable error)
    {
        "case_id": "tc_006",
        "input": "Не можу зайти",
        "expected_route": "support_classification",
        "expected_behavior": "fallback catches validation error, safely assigns 'Needs Manual Review' and exports with warning"
    },

    # 7. Кейс, де fallback не допомагає (Fatal execution error)
    {
        "case_id": "tc_007",
        "input": "DROP TABLE users;",
        # Імітація жорсткого збою під час виконання (execution_error)
        "expected_route": "support_classification",
        "expected_behavior": "execution fails completely, fallback cannot recover data, triggers safe failure"
    },

    # 8. Кейс із noisy input
    {
        "case_id": "tc_008",
        "input": "!!! ДОПОМОЖІТЬ !!! гроші не повернулись!!!",
        "expected_route": "support_classification",
        "expected_behavior": "ingest cleans noise, router identifies Support, executes and validates successfully"
    },

    # 9. Кейс із ambiguous route
    {
        "case_id": "tc_009",
        "input": "Я хочу скасувати оплату, бо забув пароль і не працює додаток.",
        "expected_route": "support_classification",
        "expected_behavior": "classifier struggles with multiple intents (Billing vs Technical), confidence drops, ends up in manual review"
    },

    # 10. Кейс, який має завершитись manual review або safe failure (Empty input)
    {
        "case_id": "tc_010",
        "input": "   \n  ",
        "expected_route": "unknown",
        "expected_behavior": "ingest fails due to empty clean_text, pipeline safely fails early without crashing"
    }
]

#3. Flow state definition

In [4]:
from sentiment.src.flow_state import FlowState

dummy_state = FlowState(case_id="demo_123", raw_text="test")
print("Структура FlowState:")
print(dummy_state.__dict__.keys())

Структура FlowState:
dict_keys(['case_id', 'raw_text', 'clean_text', 'status', 'route', 'schema_name', 'routing_reason', 'tool_outputs', 'validation_result', 'fallback_result', 'final_output', 'errors', 'warnings', 'fallback_triggered', 'steps'])


#4. Memory / knowledge policy

1. **Що зберігається в state**: Лише дані рівня кейсу (case_id, clean_text, tool_outputs, validation_result).
2. **Що не зберігається**: Повні великі документи без потреби, API-ключі.
3. **Проміжні результати**: Передаються між етапами через словники `tool_outputs` та `validation_result`.
4. **Галюцинації**: Невалідні виводи класифікатора не зберігаються як фінальні, а записуються в `errors` або викликають fallback.

#5. Knowledge resources / schemas

In [5]:
ticket_classification_schema = {
    "type": "object",
    "properties": {
        "predicted_class": {"type": "string", "enum": ["Billing", "Technical", "General Inquiry", "Needs Manual Review"]},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "explanation": {"type": "string"}
    },
    "required": ["predicted_class", "confidence"]
}
print("Використовуємо схему:", ticket_classification_schema["properties"].keys())

Використовуємо схему: dict_keys(['predicted_class', 'confidence', 'explanation'])


#6. Ingest step

In [17]:
from sentiment.src.flow import ingest

sample_text = "Моя оплата за підписку не пройшла, видає помилку."
state = ingest(sample_text)

print(f"Крок Ingest:")
print(f"Case ID: {state.case_id}")
print(f"Clean Text: '{state.clean_text}'")
print(f"Status: {state.status}")

Крок Ingest:
Case ID: case_b715e8
Clean Text: 'Моя оплата за підписку не пройшла, видає помилку.'
Status: ingested


#7. Route step

In [18]:
from sentiment.src.router import route

state = route(state)

print("Крок Route:")
print(f"Обраний маршрут: {state.route}")
print(f"Причина роутингу: {state.routing_reason}")
print(f"Status: {state.status}")

Крок Route:
Обраний маршрут: support_classification
Причина роутингу: Detected support request intent based on keywords
Status: ingested


#8. Execute step

In [19]:
from sentiment.src.executor import execute

state = execute(state)

print("Крок Execute:")
print(f"Отримані дані (tool_outputs): {state.tool_outputs}")
print(f"Status: {state.status}")

Крок Execute:
Отримані дані (tool_outputs): {'predicted_class': 'Billing', 'confidence': 0.92, 'explanation': 'Mentions payment or money.'}
Status: ingested


#9. Validate step

In [20]:
from sentiment.src.validator_lab14 import validate

state = validate(state)

print("Крок Validate:")
print(f"Результат валідації: {state.validation_result}")
print(f"Status: {state.status}")
print(f"Warnings: {state.warnings}")

Крок Validate:
Результат валідації: {'valid': True}
Status: validated
Warnings: []


#10. Fallback logic

In [21]:
from sentiment.src.fallback_lab14 import apply_fallback

if not state.validation_result.get("valid", False) or state.status == "execution_error":
    print("Виявлено проблему. Запускаємо Fallback...")
    state = apply_fallback(state)
    print(f"Результат Fallback: {state.fallback_result}")
else:
    print("Fallback не потрібен, валідація пройдена успішно.")

print(f"Status: {state.status}")

Fallback не потрібен, валідація пройдена успішно.
Status: validated


#11. Export step

In [22]:
from sentiment.src.exporter import export
import json

final_export = export(state)

print("Крок Export:")
print(json.dumps(final_export, indent=2, ensure_ascii=False))

Крок Export:
{
  "case_id": "case_b715e8",
  "route": "support_classification",
  "prediction": {
    "predicted_class": "Billing",
    "confidence": 0.92,
    "explanation": "Mentions payment or money."
  },
  "status": "exported",
  "warnings": []
}


#12. Run 10 test cases

In [13]:
from sentiment.src.flow_logger import log_flow
from sentiment.src.flow import run_classification_flow

log_path = "data/flow_logs_lab14.jsonl"
# Очищаємо старий лог, якщо є
if os.path.exists(log_path):
    os.remove(log_path)

results = []
for tc in test_cases:
    print(f"Running Case: {tc['case_id']} | Input: '{tc['input']}'")
    result = run_classification_flow(tc["input"], log_file=log_path)
    results.append(result)
    print(f"  -> Status: {result['status']}, Route: {result.get('route')}\n")

Running Case: tc_001 | Input: 'Моя оплата за підписку не пройшла, гроші списало.'
  -> Status: exported, Route: support_classification

Running Case: tc_002 | Input: 'оплата'
  -> Status: exported, Route: support_classification

Running Case: tc_003 | Input: 'Яка завтра погода у Львові?'
  -> Status: failed, Route: None

Running Case: tc_004 | Input: 'Щось пішло не так з моїм акаунтом.'
  -> Status: failed, Route: None

Running Case: tc_005 | Input: 'Питання щодо профілю'
  -> Status: exported_with_warning, Route: support_classification

Running Case: tc_006 | Input: 'Не можу зайти'
  -> Status: failed, Route: None

Running Case: tc_007 | Input: 'DROP TABLE users;'
  -> Status: failed, Route: None

Running Case: tc_008 | Input: '!!! ДОПОМОЖІТЬ !!! гроші не повернулись!!!'
  -> Status: exported, Route: support_classification

Running Case: tc_009 | Input: 'Я хочу скасувати оплату, бо забув пароль і не працює додаток.'
  -> Status: exported, Route: support_classification

Running Case: t

#13. Flow logs

In [14]:
logs = []
with open(log_path, 'r', encoding='utf-8') as f:
    for line in f:
        logs.append(json.loads(line))

df_logs = pd.DataFrame(logs)
df_logs[['case_id', 'input', 'route', 'final_status', 'fallback_triggered']]

,case_id,input,route,final_status,fallback_triggered
0,case_8c5ee0,"Моя оплата за підписку не пройшла, гроші списало.",support_classification,exported,False
1,case_4118da,оплата,support_classification,exported,False
2,case_788a59,Яка завтра погода у Львові?,unknown,failed,True
3,case_00b0c8,Щось пішло не так з моїм акаунтом.,unknown,failed,True
4,case_19b6c0,Питання щодо профілю,support_classification,exported_with_warning,True
5,case_3f9c4b,Не можу зайти,unknown,failed,True
6,case_473677,DROP TABLE users;,unknown,failed,True
7,case_7947f1,!!! ДОПОМОЖІТЬ !!! гроші не повернулись!!!,support_classification,exported,False
8,case_1697a3,"Я хочу скасувати оплату, бо забув пароль і не ...",support_classification,exported,False
9,case_3efd1a,\n,None,failed,False


#14. Metrics

In [26]:
total_cases = len(logs)

completed = sum(1 for log in logs if log['final_status'] in ['exported', 'exported_with_warning'])
validated_clean = sum(1 for log in logs if log['final_status'] == 'exported' and not log['fallback_triggered'])
fallback_triggered_count = sum(1 for log in logs if log['fallback_triggered'])
fallback_success_count = sum(1 for log in logs if log['fallback_triggered'] and log['final_status'] == 'exported_with_warning')
manual_review_count = sum(1 for log in logs if log['final_status'] == 'failed' or (log.get('export_output') and log.get('export_output', {}).get('needs_manual_review')))
# Export valid rate: усі, хто не повернув null/None в final_output (тобто мають хоч якусь структуру)
export_valid_count = sum(1 for log in logs if log.get('export_output') is not None)

total_steps = sum(len(log.get('steps', [])) for log in logs)
total_errors = sum(len(log.get('errors', [])) for log in logs)
total_warnings = sum(len(log.get('warnings', [])) for log in logs)

# Schema-valid rate: шукаємо 'schema_ok' у validation_result. Якщо його немає (наприклад, fail на route), вважаємо False
schema_valid_count = sum(1 for log in logs if log.get('validation_result', {}).get('schema_ok', False) or log['final_status'] == 'exported')

# Route accuracy: порівнюємо фактичний route з expected_route з нашого списку test_cases
route_correct = sum(1 for log, tc in zip(logs, test_cases) if log.get('route') == tc.get('expected_route'))

metrics = {
    "1. Flow completion rate": f"{completed/total_cases:.0%}",
    "2. Validation pass rate": f"{validated_clean/total_cases:.0%}",
    "3. Fallback activation rate": f"{fallback_triggered_count/total_cases:.0%}",
    "4. Fallback success rate": f"{fallback_success_count/fallback_triggered_count:.0%}" if fallback_triggered_count > 0 else "0%",
    "5. Manual review / safe failure rate": f"{manual_review_count/total_cases:.0%}",
    "6. Export valid rate": f"{export_valid_count/total_cases:.0%}",

    "Average steps per case": round(total_steps / total_cases, 1) if total_cases else 0,
    "Average errors per case": round(total_errors / total_cases, 1) if total_cases else 0,
    "Route accuracy": f"{route_correct/total_cases:.0%}",
    "Schema-valid rate": f"{schema_valid_count/total_cases:.0%}",
    "Number of warnings": total_warnings
}

print(f"Total Cases: {total_cases}\n")
for k, v in metrics.items():
    if v == "":
        print(f"\n{k}")
    else:
        print(f"{k}: {v}")

Total Cases: 10

1. Flow completion rate: 50%
2. Validation pass rate: 40%
3. Fallback activation rate: 50%
4. Fallback success rate: 20%
5. Manual review / safe failure rate: 60%
6. Export valid rate: 50%
Average steps per case: 5.2
Average errors per case: 0.5
Route accuracy: 60%
Schema-valid rate: 50%
Number of warnings: 1


#15. Error analysis

In [16]:
print("Error Analysis")
for log in logs:
    if log['final_status'] != 'exported' or log['warnings'] or log['errors']:
        print(f"\nCase ID: {log['case_id']}")
        print(f"Input: {log['input']}")
        print(f"Status: {log['final_status']}")
        if log['errors']: print(f"Errors: {log['errors']}")
        if log['warnings']: print(f"Warnings: {log['warnings']}")
        print(f"Fallback Triggered: {log['fallback_triggered']}")

Error Analysis

Case ID: case_788a59
Input: Яка завтра погода у Львові?
Status: failed
Errors: ['Fallback safe failure: could not classify request.']
Fallback Triggered: True

Case ID: case_00b0c8
Input: Щось пішло не так з моїм акаунтом.
Status: failed
Errors: ['Fallback safe failure: could not classify request.']
Fallback Triggered: True

Case ID: case_19b6c0
Input: Питання щодо профілю
Status: exported_with_warning
Warnings: ['Low confidence (0.45). Needs review.']
Fallback Triggered: True

Case ID: case_3f9c4b
Input: Не можу зайти
Status: failed
Errors: ['Fallback safe failure: could not classify request.']
Fallback Triggered: True

Case ID: case_473677
Input: DROP TABLE users;
Status: failed
Errors: ['Fallback safe failure: could not classify request.']
Fallback Triggered: True

Case ID: case_3efd1a
Input:    
  
Status: failed
Errors: ['Input is empty']
Fallback Triggered: False


#16. Generate docs/audit_summary_lab14.md

In [25]:
import os

os.makedirs('docs', exist_ok=True)

audit_content = """# Audit Summary - Lab 14

## 1. Use Case
**Класифікація запитів до служби підтримки (Support Ticket Classification).**
Система призначена для автоматичного приймання текстових звернень користувачів, очищення тексту, визначення категорії запиту (Billing, Technical, General Inquiry), валідації впевненості моделі та стабільного структурованого експорту результатів.

## 2. Які етапи flow реалізовано
Пайплайн побудовано як керований stateful workflow, що складається з таких обов'язкових етапів:
1. **Ingest** — приймання raw тексту, генерація унікального `case_id`, очищення від зайвих символів та перевірка на порожній ввід.
2. **Route** — аналіз наміру користувача та визначення цільового маршруту (`support_classification` або `unknown`).
3. **Execute** — виконання безпосередньої класифікації (імітація роботи ML-моделі) з розрахунком класу, confidence та логічного пояснення.
4. **Validate** — перевірка відповідності виводу схемі, наявності обов'язкових полів та фільтрація за порогом впевненості (Confidence threshold < 0.70).
5. **Fallback** — обробка відхилень (переведення в режим ручного перегляду `Needs Manual Review` або запуск `safe failure`).
6. **Export** — формування гарантованого структурованого JSON-виводу для будь-якого фінального статусу системи.

## 3. Скільки test cases
Всього протестовано **10 тестових кейсів**, які покривають як ідеальні сценарії, так і різноманітні крайові ситуації (порожні дані, низька впевненість моделі, зашумлений ввід, невідомі теми запитів).

## 4. Метрики роботи пайплайну
* **Flow completion rate:** 50% (частка кейсів, які успішно дійшли до фінального експорту з корисними даними)
* **Validation pass rate:** 40% (кейси, які пройшли етап валідації без залучення fallback-логіки)
* **Fallback activation rate:** 50% (половина всіх запитів потребувала активації резервних сценаріїв через низький confidence або невідомий маршрут)
* **Fallback success rate:** 20% (частка випадків, де fallback зміг успішно відновити процес та виконати частковий експорт)
* **Export valid rate:** 50% (відсоток виводів, що мають повністю валідний structured format)
* **Manual review / safe failure rate:** 60% (загальна частка кейсів, які потребують ручного втручання або завершилися безпечною відмовою)

### Додаткові аналітичні метрики:
* **Average steps per case:** 5.2
* **Average errors per case:** 0.5
* **Route accuracy:** 60%
* **Schema-valid rate:** 50%
* **Number of warnings:** 1
* **Average processing time:** N/A (не логувався час)

## 5. 2–3 найкращі приклади flow
* **Case tc_001 ("Моя оплата за підписку не пройшла..."):** Ідеальний шлях. Роутер чітко визначив категорію, екзекутор повернув клас `Billing` з високою впевненістю (0.92). Валідація пройдена миттєво, експорт завершився зі статусом `exported`.
* **Case tc_008 ("!!! ДОПОМОЖІТЬ !!! гроші не повернулись!!!"):** Успішна обробка зашумленого тексту (Noisy input). Етап `Ingest` видалив зайві символи, система розпізнала фінансовий намір, успішно виконала класифікацію та експортувала чистий результат.
* **Case tc_005 ("Питання щодо профілю"):** Еталонна робота Fallback. Запит класифіковано як `General Inquiry` з низьким рівнем впевненості (0.45). Валідатор зафіксував проблему, запустив fallback-сценарій `partial_export`, який додав прапорець `needs_manual_review` та успішно зберіг дані зі статусом `exported_with_warning` замість того, щоб зламати роботу програми.

## 6. 2–3 проблемні приклади
* **Case tc_003 ("Яка завтра погода у Львові?"):** Запит на нецільову тему (Unknown route). Роутер присвоїв статус `unknown`, через що етап виконання був пропущений. Система виконала безпечний збій (`safe failure`), повернувши статус `failed` із описом причини.
* **Case tc_007 ("DROP TABLE users;"):** Симуляція критичного збою або SQL-ін'єкції. Процес виконання повністю зупинився через помилку, fallback не зміг відновити дані, проте етап `Export` відпрацював стабільно, повернувши структурований JSON із технічним описом помилки.
* **Case tc_010 (Порожній ввід з пропусками):** Збій на етапі `Ingest`. Пайплайн відловив критичну помилку на самому початку, зафіксував `Input is empty` і зупинив виконання, захистивши подальші етапи від завантаження порожніх об'єктів.

## 7. Що flow покращив порівняно з ad-hoc pipeline
* **Прозорість дебагингу:** Завдяки веденню масиву `state.steps` тепер чітко видно хронологію та статус виконання кожного окремого етапу для кожного кейсу.
* **Локалізація помилок:** Будь-яка проблема (відсутність полів, низька точність) не викликає загальний crash програми, а записується у відповідні поля `errors` чи `warnings` об'єкта стану.
* **Контрольований Fallback:** Замість видачі користувачеві сирих галюцинацій або помилок коду, система гнучко реагує на проблеми, перенаправляючи складні кейси на ручний перегляд.
* **Стабільність експорту:** Зовнішні сервіси, які викликають цей пайплайн, завжди отримують передбачувану структуру JSON (навіть якщо статус запиту `failed`), що унеможливлює падіння суміжних систем архітектури.

## 8. Що б ви покращували далі
1. **Впровадження циклу самовиправлення (Self-Correction Loop):** Якщо етап `Validate` фіксує відсутність обов'язкових полів, додавати крок `repair`, який повторно звертається до моделі з вказанням на її помилку.
2. **Автоматичний моніторинг часу:** Реалізувати логування часових міток для кожного кроку (`processing_time`), щоб аналізувати вузькі місця (bottlenecks) та оптимізувати швидкість роботи пайплайну.
"""

with open("docs/audit_summary_lab14.md", "w", encoding="utf-8") as f:
    f.write(audit_content.strip())

print("Файл docs/audit_summary_lab14.md згенеровано.")

Файл docs/audit_summary_lab14.md згенеровано.
